# 10 — End-to-end desk questions

**Scenario.** A policy desk asks the questions that actually matter. Each run is: hybrid retrieve → ACL by role → PII redact → LLM (temperature 0) → keep only real citation ids → else abstain.

No UI. Change the question or role and re-run the cell.

This tutorial calls xAI (`XAI_API_KEY`). Swap `LLM_BASE_URL` / `LLM_MODEL` / `LLM_API_KEY` to use another OpenAI-compatible model.

What to evaluate:

- Retrieval: did the right chunks appear?
- Security: did ACL remove Restricted and MNPI chunks for the role?
- Privacy: did the prompt avoid raw synthetic PII?
- Generation: did the answer cite real chunks or abstain?

Edge cases:

- No visible evidence should produce abstention.
- A model answer without valid citations is treated as unverified.
- Public tutorials should keep notebooks output-free so users generate their own results.


Definitions and context:

| Concept | Meaning |
| --- | --- |
| End-to-end test | One question through the full RAG path |
| Citation intersection | Dropping citation IDs that were not retrieved |
| Abstention policy | Rules for when the assistant must refuse |
| Retrieval audit | Logging role, query hash, doc IDs, and chunk IDs |
| Evaluation set | Repeatable questions used to measure quality and safety |

Good RAG demos should show both successful answers and refusals. In high
security domains, "I cannot answer from visible evidence" is a correct outcome,
not a failure.


In [ ]:
import _course
from _course import HAS_LLM
from lib.pipeline import generate, load_index, retrieve

index = load_index()
print("chunks:", len(index.chunks))
if not HAS_LLM:
    print("No LLM key — retrieval only. Set XAI_API_KEY to generate answers.")


def show(question, role):
    print("=" * 72)
    print("Q:", question)
    print("role:", role)
    hits = retrieve(index, question, role)
    for h in hits:
        print(f"  {h.chunk.source_type:5} {h.chunk.classification:14} {h.chunk.doc_id} / {h.chunk.section}")
    if not HAS_LLM:
        return
    result = generate(question, role, hits)
    print("abstained:", result["abstained"])
    print(result["answer"])
    print("citations:", result.get("citation_ids"))


In [ ]:
show("What information classification labels does Northline use?", "employee")


In [ ]:
show("What severity is a full TIN in an unsanctioned chatbot?", "employee")


In [ ]:
show("How long do we keep retrieval audit logs?", "employee")


In [ ]:
show("What is Morgan Ellison's custody account number?", "advisor")


In [ ]:
show("What is the break-glass quorum and who holds cubbies?", "advisor")
show("What is the break-glass quorum and who holds cubbies?", "security")


In [ ]:
show("What is Watch Book entry WB-8841?", "advisor")
show("What is Watch Book entry WB-8841?", "compliance")


In [ ]:
show("Can I paste Confidential data into any internal chatbot? Is Sensitive still a label?", "employee")


That is the pipeline. Retrieval decides access. The model only writes from what is left.
